<a href="https://colab.research.google.com/github/msharma962/AI-RL-Repo/blob/main/Huggy_the_dog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Clone the repository (can take 3min)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

In [ ]:
!python --version

In [ ]:
import os
import re

setup_paths = [
    '/content/ml-agents/ml-agents/setup.py',
    '/content/ml-agents/ml-agents-envs/setup.py'
]

for path in setup_paths:
    if os.path.exists(path):
        print(f"Checking: {path}")
        with open(path, 'r') as f:
            content = f.read()
            match = re.search(r"python_requires\s*=\s*['\"]([^'\"]+)['\"]", content)
            if match:
                print(f"  Required Python Version: {match.group(1)}")
            else:
                print("  Could not find explicit 'python_requires' setting in this file.")
    else:
        print(f"File not found: {path}")

In [ ]:
# As, googlecolab's python version is 3.23.16 and the ML-Agent supports >=3.10.2 and <=3.10.12
# We will create a virtual env
# Install virtualenv and create a virtual environment
!pip install virtualenv
!virtualenv myenv

# Download and install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Activate Miniconda and install Python ver 3.10.12
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # Specify the version here

# Set environment variables for Python and conda paths
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv


In [ ]:
# 1. Accept Anaconda's Terms of Service so packages can download
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# 2. Create a clean conda environment with the exact Python version needed for ML-Agents
!conda create -y -n mlagents_env python=3.10.12

In [ ]:
# 3. Verify the Python version by executing the Python binary inside the conda environment directly
!/usr/local/envs/mlagents_env/bin/python --version

In [ ]:
# Change directory to the cloned ml-agents repository
%cd /content/ml-agents

# Install the sub-packages using the pip belonging to the python 3.10 environment
!/usr/local/envs/mlagents_env/bin/pip install -e ./ml-agents-envs
!/usr/local/envs/mlagents_env/bin/pip install -e ./ml-agents

In [ ]:
# Ensure we are in the ml-agents directory and create the required subdirectories
%cd /content/ml-agents
!mkdir -p ./trained-envs-executables/linux

# List the directory structure to verify it was created successfully
!ls -d ./trained-envs-executables/linux

In [ ]:
%cd /content/ml-agents
# Download Huggy.zip
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip


In [ ]:
# perform all operations in /content/ml-agent
%pwd

In [ ]:
%%capture
# Unzip Huggy environment executable into the destination directory
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip

In [ ]:
!chmod -R 755 ./trained-envs-executables/linux/Huggy

In [ ]:
%pwd

In [ ]:
%cd /content/ml-agents
!ls -la config/

In [ ]:
%cd /content/ml-agents/config/ppo
!ls -la

In [ ]:
import os
print("Current Python Working Directory:", os.getcwd())

In [ ]:
# vi not supported by Google colab
# Instead use %%writefile jupiter command
%%writefile Huggy.yaml
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

In [ ]:
# verify the file you created is saved and present
!cat Huggy.yaml

In [ ]:
# Change working directory back to the repository root
%cd /content/ml-agents

# Resume training using the --resume flag
!/usr/local/envs/mlagents_env/bin/mlagents-learn ./config/ppo/Huggy.yaml --env=./trained-envs-executables/linux/Huggy/Huggy --run-id="Huggy" --no-graphics --resume